To test using OpenAI to search PDF to see if it's better than my own RAG

In [15]:
pip install openai PyMuPDF


In [16]:
pip install --upgrade openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 599.1/599.1 kB 18.7 MB/s eta 0:00:00
  Attempting uninstall: openai
    Found existing installation: openai 1.68.2
    Uninstalling openai-1.68.2:
      Successfully uninstalled openai-1.68.2


In [2]:
import os
# from utils import get_openai_api_key
from google.colab import userdata

os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')
os.environ["OPENAI_MODEL_NAME"] = 'gpt-4o-mini'
os.environ["SERPER_API_KEY"] = userdata.get('SERPER_API_KEY')

In [7]:
import requests
import os

def download_file(url: str, destination_folder: str = ".") -> str:
    """
    Downloads a file from the given URL and saves it locally.

    Args:
        url (str): The URL of the file to download.
        destination_folder (str, optional): Where to save the file. Defaults to current directory.

    Returns:
        str: Full path to the downloaded file or an error message.
    """
    try:
        os.makedirs(destination_folder, exist_ok=True)

        # Extract base filename from URL
        original_filename = url.split("/")[-1]
        base_name, ext = os.path.splitext(original_filename)
        destination_path = os.path.join(destination_folder, original_filename)

        # Handle filename conflict by appending a number
        counter = 1
        while os.path.exists(destination_path):
            destination_path = os.path.join(destination_folder, f"{base_name}_{counter}{ext}")
            counter += 1

        # Download the file
        response = requests.get(url)
        response.raise_for_status()
        with open(destination_path, 'wb') as f:
            f.write(response.content)

        return f"{destination_path}"
    except Exception as e:
        return f"Failed to download file from {url}. Error: {str(e)}"

In [5]:
import fitz  # PyMuPDF

# Function to extract text from PDF
def extract_text_from_pdf(pdf_path):
    doc = fitz.open(pdf_path)
    text = ''
    for page in doc:
        text += page.get_text()
    return text

In [22]:
import openai

local_file1 = download_file("https://reports.adviserinfo.sec.gov/reports/ADV/124982/PDF/124982.pdf")

# Load and extract both PDFs
pdf1_text = extract_text_from_pdf(local_file1)


# Create prompt
prompt = f"""
Analyze this SEC ADV Form and list all direct owners and executive officers from Schedule A.
- include Full legal name, Title and ownership code and description. For instance, for code 'C', the description is '25% but less than 50%'.
- return in JSON format.

Detailed ADV Form is as follows:
{pdf1_text}  # Truncate to avoid token limits
"""

client=openai.OpenAI()
# API call (NEW SYNTAX)
response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": "You are an expert financial analyst."},
        {"role": "user", "content": prompt}
    ],
    temperature=0.1,
    max_tokens=1000
)

# Print response
# Accessing the content of the first choice from the response
print(response.choices[0].message.content) # changed from response.choices[0].message.content

```json
{
  "owners_and_executive_officers": [
    {
      "full_legal_name": "BUTTERLY, WILLIAM, GEORGE",
      "title": "GENERAL COUNSEL, DIRECTOR OF SUSTAINABILITY & ENGAGEMENT, & SECRETARY",
      "ownership_code": "NA",
      "ownership_description": "less than 5%"
    },
    {
      "full_legal_name": "OCE US HOLDING, INC.",
      "title": "SOLE SHAREHOLDER",
      "ownership_code": "E",
      "ownership_description": "75% or more"
    },
    {
      "full_legal_name": "FEENEY, JOSEPH, FRANCIS",
      "title": "CHIEF EXECUTIVE OFFICER & DIRECTOR",
      "ownership_code": "NA",
      "ownership_description": "less than 5%"
    },
    {
      "full_legal_name": "DONOVAN, MARK, EDWARD",
      "title": "DIRECTOR",
      "ownership_code": "NA",
      "ownership_description": "less than 5%"
    },
    {
      "full_legal_name": "VARNER, GREG, ALLAN",
      "title": "CHIEF FINANCIAL OFFICER & TREASURER",
      "ownership_code": "NA",
      "ownership_description": "less than 5%"
    },
